# Analisis de repositorios

In [121]:
from pathlib import Path

import pandas as pd
import numpy as np
import json

In [122]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", None)

RAW_DIR = Path("../data/raw").resolve()

print("Ruta de datos:")
print(RAW_DIR)

print("\n¿Existe?")
print(RAW_DIR.exists())

Ruta de datos:
/Users/uriel/Documents/machine_learning/data/raw

¿Existe?
True


In [123]:
repositorios = [
    carpeta
    for carpeta in RAW_DIR.iterdir()
    if carpeta.is_dir()
]

for repo in repositorios:
    print(repo.name)
    

03
05
02
01


In [124]:
for repo in repositorios:

    print("\n" + "=" * 100)
    print(f"REPOSITORIO: {repo.name}")
    print("=" * 100)

    archivos = [
        archivo
        for archivo in repo.rglob("*")
        if archivo.is_file()
    ]

    print(f"Total de archivos: {len(archivos):,}")

    extensiones = {}

    for archivo in archivos:
        ext = archivo.suffix.lower()
        extensiones[ext] = extensiones.get(ext, 0) + 1

    print("Tipos de archivo:")

    for extension, cantidad in sorted(extensiones.items()):
        print(f"{extension or '[sin extensión]'}: {cantidad:,}")


REPOSITORIO: 03
Total de archivos: 1
Tipos de archivo:
.csv: 1

REPOSITORIO: 05
Total de archivos: 2
Tipos de archivo:
.csv: 2

REPOSITORIO: 02
Total de archivos: 13,878
Tipos de archivo:
.html: 857
.json: 13,021

REPOSITORIO: 01
Total de archivos: 3
Tipos de archivo:
[sin extensión]: 1
.csv: 2


In [125]:
registros = []

for archivo in RAW_DIR.rglob("*"):

    if not archivo.is_file():
        continue

    # No incluimos nuestros reportes de copia
    if archivo.name in {
        "MANIFEST_COPIA.csv",
        "PLAN_COPIA.csv",
        "ERRORES_COPIA.csv"
    }:
        continue

    relativa = archivo.relative_to(RAW_DIR)

    registros.append({
        "repositorio": relativa.parts[0],
        "archivo": archivo.name,
        "extension": archivo.suffix.lower(),
        "ruta_relativa": str(relativa),
        "tamano_mb": archivo.stat().st_size / (1024 ** 2)
    })

df_archivos = pd.DataFrame(registros)

df_archivos.head(20)

,repositorio,archivo,extension,ruta_relativa,tamano_mb
0,.DS_Store,.DS_Store,,.DS_Store,0.007816
1,01,.DS_Store,,01/.DS_Store,0.005863
2,01,aggregates.csv,.csv,01/28489601/aggregates.csv,6721.329556
3,01,interactions.csv,.csv,01/28489601/interactions.csv,3603.754929
4,02,1_Code.exe_1764911116433_a11y_tree.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/59/1_Code.exe_1764911116433_a11y_tree.json,0.004945
5,02,metadata_59.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/59/metadata_59.json,0.001658
6,02,1_Code.exe.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/59/1_Code.exe.json,0.037566
7,02,metadata_50.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/50/metadata_50.json,0.001404
8,02,metadata_57.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/57/metadata_57.json,0.001359
9,02,metadata_32.json,.json,02/Reduced-A11y-CUA/Qwen3-VL-32B-Instruct/Screen Reader/32/metadata_32.json,0.001776


In [126]:
df_archivos.shape

(13885, 5)

In [127]:
df_archivos.groupby(
    ["repositorio", "extension"]
).size().reset_index(
    name="cantidad"
)

,repositorio,extension,cantidad
0,.DS_Store,,1
1,01,,1
2,01,.csv,2
3,02,.html,857
4,02,.json,13021
5,03,.csv,1
6,05,.csv,2


## REPOSITORIO 1

In [128]:
repo_01 = RAW_DIR / "01" / "28489601"

aggregates_path = repo_01 / "aggregates.csv"
interactions_path = repo_01 / "interactions.csv"

In [129]:
aggregates_sample = pd.read_csv(
    aggregates_path,
    nrows=1000,
    low_memory=False
)

print("Dimensiones de la muestra:")
print(aggregates_sample.shape)

aggregates_sample.head()

Dimensiones de la muestra:
(1000, 1235)


,middle.id,middle.user_id,middle.timestamp,middle.url,middle.mouse.position.x,middle.mouse.position.y,middle.mouse.clicks,middle.mouse.clicks.left,middle.mouse.clicks.right,middle.mouse.clicks.middle,middle.mouse.clicks.others,middle.mouse.speed,middle.mouse.speed.x,middle.mouse.speed.y,middle.mouse.acceleration,middle.mouse.acceleration.x,middle.mouse.acceleration.y,middle.trajectory.slope,middle.scroll.absolute.x,middle.scroll.absolute.y,middle.scroll.relative.x,middle.scroll.relative.y,middle.keyboard,middle.keyboard.alpha,middle.keyboard.numeric,middle.keyboard.function,middle.keyboard.symbol,middle.emotions.automatic_sampling,middle.emotions.exist,middle.emotions.joy,middle.emotions.fear,middle.emotions.disgust,middle.emotions.sadness,middle.emotions.anger,middle.emotions.surprise,middle.emotions.contempt,middle.emotions.valence,middle.emotions.engagement,25.full.clicks.all.sum,25.full.clicks.all.avg,25.full.clicks.all.std,25.full.clicks.left.sum,25.full.clicks.left.avg,25.full.clicks.left.std,25.full.clicks.middle.sum,25.full.clicks.middle.avg,25.full.clicks.middle.std,25.full.clicks.right.sum,25.full.clicks.right.avg,25.full.clicks.right.std,25.full.clicks.other.sum,25.full.clicks.other.avg,25.full.clicks.other.std,25.full.event_times.sum,25.full.event_times.avg,25.full.event_times.std,25.full.idle.sum,25.full.idle.avg,25.full.idle.std,25.full.keys.all.sum,25.full.keys.all.avg,25.full.keys.all.std,25.full.keys.alphabetic.sum,25.full.keys.alphabetic.avg,25.full.keys.alphabetic.std,25.full.keys.numeric.sum,25.full.keys.numeric.avg,25.full.keys.numeric.std,25.full.keys.symbol.sum,25.full.keys.symbol.avg,25.full.keys.symbol.std,25.full.keys.function.sum,25.full.keys.function.avg,25.full.keys.function.std,25.full.keys.alphanumeric.sum,25.full.keys.alphanumeric.avg,25.full.keys.alphanumeric.std,25.full.mouse_movements.rate,25.full.mouse_movements.total,25.full.scrolls.rate,25.full.scrolls.total,25.full.slopes.changes,25.full.slopes.change_rate,25.full.urls.unique,25.full.urls.changed,25.full.urls.change_rate,25.before.clicks.all.sum,25.before.clicks.all.avg,25.before.clicks.all.std,25.before.clicks.left.sum,25.before.clicks.left.avg,25.before.clicks.left.std,25.before.clicks.middle.sum,25.before.clicks.middle.avg,25.before.clicks.middle.std,25.before.clicks.right.sum,25.before.clicks.right.avg,25.before.clicks.right.std,25.before.clicks.other.sum,25.before.clicks.other.avg,25.before.clicks.other.std,25.before.event_times.sum,25.before.event_times.avg,25.before.event_times.std,25.before.idle.sum,25.before.idle.avg,25.before.idle.std,25.before.keys.all.sum,25.before.keys.all.avg,25.before.keys.all.std,25.before.keys.alphabetic.sum,25.before.keys.alphabetic.avg,25.before.keys.alphabetic.std,25.before.keys.numeric.sum,25.before.keys.numeric.avg,25.before.keys.numeric.std,25.before.keys.symbol.sum,25.before.keys.symbol.avg,25.before.keys.symbol.std,25.before.keys.function.sum,25.before.keys.function.avg,25.before.keys.function.std,25.before.keys.alphanumeric.sum,25.before.keys.alphanumeric.avg,25.before.keys.alphanumeric.std,25.before.mouse_movements.rate,25.before.mouse_movements.total,25.before.scrolls.rate,25.before.scrolls.total,25.before.slopes.changes,25.before.slopes.change_rate,25.before.urls.unique,25.before.urls.changed,25.before.urls.change_rate,25.after.clicks.all.sum,25.after.clicks.all.avg,25.after.clicks.all.std,25.after.clicks.left.sum,25.after.clicks.left.avg,25.after.clicks.left.std,25.after.clicks.middle.sum,25.after.clicks.middle.avg,25.after.clicks.middle.std,25.after.clicks.right.sum,25.after.clicks.right.avg,25.after.clicks.right.std,25.after.clicks.other.sum,25.after.clicks.other.avg,25.after.clicks.other.std,25.after.event_times.sum,25.after.event_times.avg,25.after.event_times.std,25.after.idle.sum,25.after.idle.avg,25.after.idle.std,25.after.keys.all.sum,25.after.keys.all.avg,25.after.keys.all.std,25.after.keys.alphabetic.sum,25.after.keys.alphabetic.avg,25.after.keys.alphabetic.std,25.after.keys.numeric

In [130]:
pd.DataFrame({
    "variable": aggregates_sample.columns,
    "tipo": aggregates_sample.dtypes.astype(str).values,
    "nulos_muestra": aggregates_sample.isna().sum().values,
    "valores_unicos_muestra": [
        aggregates_sample[col].nunique()
        for col in aggregates_sample.columns
    ]
})

,variable,tipo,nulos_muestra,valores_unicos_muestra
0,middle.id,str,0,1000
1,middle.user_id,str,0,1
2,middle.timestamp,int64,0,996
3,middle.url,str,0,18
4,middle.mouse.position.x,int64,0,116
...,...,...,...,...
1230,2000.after.speed.x.avg,float64,0,93
1231,2000.after.speed.x.std,float64,0,93
1232,2000.after.speed.y.sum,float64,0,87
1233,2000.after.speed.y.avg,float64,0,94


In [131]:
aggregates_sample.describe().T

,count,mean,std,min,25%,50%,75%,max
middle.timestamp,1000.0,1.737089e+12,2.266378e+07,1.737044e+12,1.737098e+12,1.737099e+12,1.737102e+12,1.737104e+12
middle.mouse.position.x,1000.0,9.978320e+02,4.544025e+02,0.000000e+00,5.750000e+02,1.131000e+03,1.393000e+03,1.536000e+03
middle.mouse.position.y,1000.0,3.801630e+02,2.378656e+02,0.000000e+00,2.000000e+02,4.430000e+02,5.367500e+02,8.350000e+02
middle.mouse.speed,1000.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
middle.mouse.speed.x,1000.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...
2000.after.speed.x.avg,1000.0,1.073497e-01,1.474118e+00,-1.620361e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.487179e+01
2000.after.speed.x.std,1000.0,3.706393e+01,6.114685e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.463907e+04
2000.after.speed.y.sum,1000.0,4.165745e-02,4.085885e+00,-4.698153e+01,0.000000e+00,0.000000e+00,0.000000e+00,2.750000e+01
2000.after.speed.y.avg,1000.0,3.740733e-03,1.362805e-01,-9.396306e-01,0.000000e+00,0.000000e+00,0.000000e+00,2.115385e+00


In [132]:
interactions_sample = pd.read_csv(
    interactions_path,
    nrows=1000,
    low_memory=False
)

print("Dimensiones de la muestra:")
print(interactions_sample.shape)

interactions_sample.head()

Dimensiones de la muestra:
(1000, 38)


,id,user_id,timestamp,url,mouse.position.x,mouse.position.y,mouse.clicks,mouse.clicks.left,mouse.clicks.right,mouse.clicks.middle,mouse.clicks.others,mouse.speed,mouse.speed.x,mouse.speed.y,mouse.acceleration,mouse.acceleration.x,mouse.acceleration.y,trajectory.slope,scroll.absolute.x,scroll.absolute.y,scroll.relative.x,scroll.relative.y,keyboard,keyboard.alpha,keyboard.numeric,keyboard.function,keyboard.symbol,emotions.automatic_sampling,emotions.exist,emotions.joy,emotions.fear,emotions.disgust,emotions.sadness,emotions.anger,emotions.surprise,emotions.contempt,emotions.valence,emotions.engagement
0,67892e1a85dffb571207298e,678927b385dffb57120720c3,1737043478562,https://www.repubblica.it/,0,0,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,400.0,100.0,400.0,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,67892e1a85dffb571207298f,678927b385dffb57120720c3,1737043478685,https://www.repubblica.it/,0,0,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,400.0,100.0,400.0,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,67892e1a85dffb5712072990,678927b385dffb57120720c3,1737043478810,https://www.repubblica.it/,0,0,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,400.0,100.0,400.0,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,67892e1a85dffb5712072991,678927b385dffb57120720c3,1737043478857,https://www.repubblica.it/,0,0,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,400.0,100.0,400.0,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,67892e1a85dffb5712072992,678927b385dffb57120720c3,1737043478966,https://www.repubblica.it/,0,0,False,False,False,False,False,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,400.0,100.0,400.0,False,False,False,False,False,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [133]:
pd.DataFrame({
    "variable": interactions_sample.columns,
    "tipo": interactions_sample.dtypes.astype(str).values,
    "nulos_muestra": interactions_sample.isna().sum().values,
    "valores_unicos_muestra": [
        interactions_sample[col].nunique()
        for col in interactions_sample.columns
    ]
})

,variable,tipo,nulos_muestra,valores_unicos_muestra
0,id,str,0,1000
1,user_id,str,0,1
2,timestamp,int64,0,996
3,url,str,0,3
4,mouse.position.x,int64,0,269
5,mouse.position.y,int64,0,264
6,mouse.clicks,bool,0,2
7,mouse.clicks.left,bool,0,2
8,mouse.clicks.right,bool,0,1
9,mouse.clicks.middle,bool,0,1


## REPOSITORIO 2

In [134]:
a11y_root = RAW_DIR / "02" / "Reduced-A11y-CUA"

metadata_files = list(
    a11y_root.rglob("metadata_*.json")
)

a11y_tree_files = list(
    a11y_root.rglob("*_a11y_tree.json")
)

web_log_json_files = [
    archivo
    for archivo in a11y_root.rglob("*.json")
    if "web_logs" in archivo.parts
    and "_a11y_tree" not in archivo.name
]

web_html_files = list(
    a11y_root.rglob("*.html")
)

print(f"Metadata:            {len(metadata_files):,}")
print(f"Accessibility Tree:  {len(a11y_tree_files):,}")
print(f"Web logs JSON:       {len(web_log_json_files):,}")
print(f"HTML:                {len(web_html_files):,}")

Metadata:            1,320
Accessibility Tree:  7,419
Web logs JSON:       1,329
HTML:                857


In [135]:
def cargar_metadata_a11y(archivos):

    registros = []

    for archivo in archivos:

        try:

            with open(
                archivo,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

            fila = pd.json_normalize(
                data,
                sep="."
            )

            fila["source_file"] = str(
                archivo.relative_to(a11y_root)
            )

            # Extraemos información de la ruta
            relativa = archivo.relative_to(a11y_root)

            if len(relativa.parts) > 0:
                fila["grupo"] = relativa.parts[0]

            registros.append(fila)

        except Exception as e:

            print(
                f"Error leyendo {archivo}: {e}"
            )

    return pd.concat(
        registros,
        ignore_index=True
    )

In [136]:
a11y_metadata = cargar_metadata_a11y(
    metadata_files
)

In [137]:
a11y_metadata.shape

(1320, 38)

In [138]:
a11y_metadata.head()

,task.task_id,task.task_title,task.instruction,task.context,task.task_group,task.familiarity,task.difficulty_before,task.difficulty_after,task.success,session.started_at,session.ended_at,session.folder,session.applications,accessibility.baseline.os,accessibility.baseline.os_version,accessibility.baseline.python_version,accessibility.baseline.narrator.enabled,accessibility.baseline.narrator.startup_enabled,accessibility.baseline.magnifier.enabled,accessibility.baseline.magnifier.zoom,accessibility.baseline.color_filter.enabled,accessibility.baseline.color_filter.type,accessibility.baseline.high_contrast.enabled,accessibility.baseline.high_contrast.flags,accessibility.baseline.sticky_keys.flags,accessibility.baseline.sticky_keys.maybe_enabled,accessibility.baseline.toggle_keys.flags,accessibility.baseline.toggle_keys.maybe_enabled,accessibility.baseline.filter_keys.flags,accessibility.baseline.filter_keys.maybe_enabled,accessibility.baseline.font_smoothing,accessibility.baseline.display_scaling,accessibility.baseline.mouse_cursor_scheme,accessibility.changes,source_file,grupo,task.failure_reason,task.failure_notes
0,59,Enable Picture-in-Picture playback,"Find ""Click This Video To Feed 1 Person"" by Mr Beast on YouTube. Enable picture-in-picture playback on Youtube.",I want a video I am watching on YouTube to float in a small window while I look for other videos.,Media,High,Low,Medium,False,1.764911e+09,1.764912e+09,task_logs\User 24\89,"[{'order': 1, 'app': 'Code.exe', 'log_file': '1_Code.exe.json', 'a11y_tree_files': ['1_Code.exe_1764911116433_a11y_tree.json'], 'events': 50, 'tab...",Windows,10.0.26100,3.13.7,False,False,False,None,False,unknown,False,126,510,True,62,True,126,True,2,None,windows aero,[],Qwen3-VL-32B-Instruct/Screen Reader/59/metadata_59.json,Qwen3-VL-32B-Instruct,NaN,NaN
1,50,Change the video speed,Open video Task ID 75.mp4 in Documents > Task 75 folder on Media Player. Play the video in 1.5x speed.,"I'm watching a video on Media Player, but I'm on a time crunch so I want to speed through it.",Media,High,Low,Medium,False,1.764907e+09,1.764908e+09,task_logs\User 24\75,[],Windows,10.0.26100,3.13.7,False,False,False,None,False,unknown,False,126,510,True,62,True,126,True,2,None,windows aero,[],Qwen3-VL-32B-Instruct/Screen Reader/50/metadata_50.json,Qwen3-VL-32B-Instruct,NaN,NaN
2,57,Change the language of the video,"Find ""Click This Video To Feed 1 Person"" by Mr Beast on YouTube. Change the audio track to Spanish instead.","I'm watching a YouTube video, but this video is in English and I want to improve my Spanish ability.",Media,High,Low,NaN,False,1.764908e+09,NaN,task_logs\User 24\87,[],Windows,10.0.26100,3.13.7,False,False,False,None,False,unknown,False,126,510,True,62,True,126,True,2,None,windows aero,[],Qwen3-VL-32B-Instruct/Screen Reader/57/metadata_57.json,Qwen3-VL-32B-Instruct,NaN,NaN
3,32,Add a strike-through and highlight to text in Word,"Open Task ID 46.docx in Documents > Task 46 folder, make a strikethrough to the title of the article and highlight them.",I'm reading a word document. I want to suggest changes to the article's title by striking out the title from the document so that the author is aw...,Document Editing,High,Low,Medium,False,1.764898e+09,1.764899e+09,task_logs\User 24\46,"[{'order': 1, 'app': 'Code.exe', 'log_file': '1_Code.exe.json', 'a11y_tree_files': ['1_Code.exe_1764898115564_a11y_tree.json'], 'events': 49, 'tab...",Windows,10.0.26100,3.13.7,False,False,False,None,False,unknown,False,126,510,True,62,True,126,True,2,None,windows aero,[],Qwen3-VL-32B-Instruct/Screen Reader/32/metadata_32.json,Qwen3-VL-32B-Instruct,NaN,NaN
4,35,Capitalize the starting letter of every word in Word and add page number,"Open Task ID 50.docx in Document > Task 50 folder, capitalize the first letter of every word in the title and add the page number",I am writing an article on Word and I want to modify the title format based on the guidelines.,Document Editing,High,Low,Medium,False,1.764902e+09,1.764903e

In [139]:
a11y_metadata.columns.tolist()

['task.task_id',
 'task.task_title',
 'task.instruction',
 'task.context',
 'task.task_group',
 'task.familiarity',
 'task.difficulty_before',
 'task.difficulty_after',
 'task.success',
 'session.started_at',
 'session.ended_at',
 'session.folder',
 'session.applications',
 'accessibility.baseline.os',
 'accessibility.baseline.os_version',
 'accessibility.baseline.python_version',
 'accessibility.baseline.narrator.enabled',
 'accessibility.baseline.narrator.startup_enabled',
 'accessibility.baseline.magnifier.enabled',
 'accessibility.baseline.magnifier.zoom',
 'accessibility.baseline.color_filter.enabled',
 'accessibility.baseline.color_filter.type',
 'accessibility.baseline.high_contrast.enabled',
 'accessibility.baseline.high_contrast.flags',
 'accessibility.baseline.sticky_keys.flags',
 'accessibility.baseline.sticky_keys.maybe_enabled',
 'accessibility.baseline.toggle_keys.flags',
 'accessibility.baseline.toggle_keys.maybe_enabled',
 'accessibility.baseline.filter_keys.flags',
 'a

In [140]:
pd.DataFrame({
    "variable": a11y_metadata.columns,
    "tipo": a11y_metadata.dtypes.astype(str).values,
    "nulos": a11y_metadata.isna().sum().values,
    "porcentaje_nulos": (
        a11y_metadata.isna().mean().values * 100
    ).round(2)
})

,variable,tipo,nulos,porcentaje_nulos
0,task.task_id,int64,0,0.00
1,task.task_title,str,0,0.00
2,task.instruction,str,0,0.00
3,task.context,str,0,0.00
4,task.task_group,str,0,0.00
5,task.familiarity,str,0,0.00
6,task.difficulty_before,str,0,0.00
7,task.difficulty_after,str,12,0.91
8,task.success,bool,0,0.00
9,session.started_at,float64,0,0.00


In [141]:
archivo = a11y_tree_files[0]

with open(
    archivo,
    "r",
    encoding="utf-8"
) as f:

    a11y_example = json.load(f)

print(type(a11y_example))

<class 'dict'>


In [142]:
a11y_example.keys()

dict_keys(['name', 'control_type', 'automation_id', 'bounding_rectangle', 'children'])

In [143]:
a11y_tree_sample = pd.json_normalize(
    a11y_example,
    sep="."
)

a11y_tree_sample.head()

,name,control_type,automation_id,children,bounding_rectangle.left,bounding_rectangle.top,bounding_rectangle.right,bounding_rectangle.bottom
0,readme.md - computer-use-agents - Visual Studio Code,PaneControl,,"[{'name': 'readme.md - computer-use-agents - Visual Studio Code', 'control_type': 'DocumentControl', 'automation_id': '841024', 'bounding_rectangl...",93,0,1863,1008


In [144]:
a11y_tree_sample.columns.tolist()

['name',
 'control_type',
 'automation_id',
 'children',
 'bounding_rectangle.left',
 'bounding_rectangle.top',
 'bounding_rectangle.right',
 'bounding_rectangle.bottom']

In [145]:
archivo = web_log_json_files[0]

with open(
    archivo,
    "r",
    encoding="utf-8"
) as f:

    web_log_example = json.load(f)

print(type(web_log_example))

<class 'dict'>


In [146]:
if isinstance(web_log_example, dict):
    print(web_log_example.keys())

elif isinstance(web_log_example, list):
    print(f"Número de elementos: {len(web_log_example)}")

    if len(web_log_example) > 0:
        print(web_log_example[0])

dict_keys(['url', 'title', 'order', 'created_at', 'dom_file', 'dom_url', 'a11y_file', 'a11y_url', 'interactions'])


In [147]:
web_log_sample = pd.json_normalize(
    web_log_example,
    sep="."
)

web_log_sample.head()

,url,title,order,created_at,dom_file,dom_url,a11y_file,a11y_url,interactions
0,https://www.apple.com/,Apple,1,1764844377283,task_logs/User 17/5/web_logs/web_tab1_1764844377283.html,/task_logs/User 17/5/web_logs/web_tab1_1764844377283.html,task_logs/User 17/5/web_logs/web_tab1_1764844377283_a11y_tree.json,/task_logs/User 17/5/web_logs/web_tab1_1764844377283_a11y_tree.json,"[{'type': 'focus', 'timestamp': 1764844377282, 'selector': 'a.globalnav-link.globalnav-submenu-trigger-link.globalnav-link-store', 'element': {'ta..."


In [148]:
web_log_sample.columns.tolist()

['url',
 'title',
 'order',
 'created_at',
 'dom_file',
 'dom_url',
 'a11y_file',
 'a11y_url',
 'interactions']

## REPOSITORIO 3

In [149]:
prototype_path = (
    RAW_DIR
    / "03"
    / "29042675"
    / "Prototype_Metadata.csv"
)

prototype_metadata = pd.read_csv(
    prototype_path
)

prototype_metadata.head()

,Prototype,Prototype Type,Domain,Scource URL
0,Food Order,High Fidelity,Delivery,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0
1,Gaira (Community app),High Fidelity,Community,https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0?
2,E-Tutor,High Fidelity,Education,https://www.figma.com/community/file/1271304360859402063
3,Tracking App,High Fidelity,Mobile app,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects
4,Travel,High Fidelity,Travel,https://www.figma.com/community/file/1379689864793179441


In [150]:
prototype_metadata.shape

(20, 4)

In [151]:
prototype_metadata.columns.tolist()

['Prototype', 'Prototype Type', 'Domain', 'Scource URL']

In [152]:
prototype_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Prototype       20 non-null     str  
 1   Prototype Type  20 non-null     str  
 2   Domain          20 non-null     str  
 3   Scource URL     20 non-null     str  
dtypes: str(4)
memory usage: 772.0 bytes


In [153]:
prototype_metadata.describe(
    include="all"
).T

,count,unique,top,freq
Prototype,20,19,Bakery,2
Prototype Type,20,2,High Fidelity,10
Domain,20,12,Healthcare,5
Scource URL,20,20,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App--Community-?node-id=0-1&t=h3GPGrBwJYFFbxI3-0,1


## REPOSITORIO 5

In [168]:
repo_05 = RAW_DIR / "05" / "delfi22-main" / "data"

m_assign = pd.read_csv(
    repo_05 / "m_assign.csv",
    sep=";",
    low_memory=False
)

m_quiz = pd.read_csv(
    repo_05 / "m_quiz.csv",
    sep=";",
    low_memory=False
)

print("m_assign:", m_assign.shape)
print("m_quiz:", m_quiz.shape)

display(m_assign.head())
display(m_quiz.head())

m_assign: (68, 29)
m_quiz: (158, 41)


,id,course,name,intro,introformat,alwaysshowdescription,nosubmissions,submissiondrafts,sendnotifications,sendlatenotifications,duedate,allowsubmissionsfromdate,grade,timemodified,requiresubmissionstatement,completionsubmit,cutoffdate,gradingduedate,teamsubmission,requireallteammemberssubmit,teamsubmissiongroupingid,blindmarking,revealidentities,attemptreopenmethod,maxattempts,markingworkflow,markingallocation,sendstudentnotifications,preventsubmissionnotingroup
0,4,2,Einsendeaufgabe 1.1,<removed>,1,1,0,0,0,0,1572303540,1569276000,6,1571305062,0,1,1572303540,0,0,0,0,0,0,none,-1,0,0,1,0
1,20,2,Einsendeaufgabe 2.8,<removed>,1,1,0,0,0,0,1574722740,1569276000,8,1569400539,0,1,1574722740,0,0,0,0,0,0,none,-1,0,0,1,0
2,41,5,Einsendeaufgabe 1.2,<removed>,1,1,0,0,0,0,1603666740,1600984800,5,1599809892,0,1,1603753140,0,0,0,0,0,0,none,-1,0,0,1,0
3,30,2,Einsendeaufgabe 3.7,<removed>,1,1,0,0,0,0,1578351540,1569362400,10,1569400782,0,1,1578351540,0,0,0,0,0,0,none,-1,0,0,1,0
4,11,2,Einsendeaufgabe 1.8,<removed>,1,1,0,0,0,0,1572303540,1569276000,14,1569400337,0,1,1572303540,0,0,0,0,0,0,none,-1,0,0,1,0


,id,course,name,intro,introformat,timeopen,timeclose,timelimit,overduehandling,graceperiod,preferredbehaviour,canredoquestions,attempts,attemptonlast,grademethod,decimalpoints,questiondecimalpoints,reviewattempt,reviewcorrectness,reviewmarks,reviewspecificfeedback,reviewgeneralfeedback,reviewrightanswer,reviewoverallfeedback,questionsperpage,navmethod,shuffleanswers,sumgrades,grade,timecreated,timemodified,password,subnet,browsersecurity,delay1,delay2,showuserpicture,showblocks,completionattemptsexhausted,completionpass,allowofflineattempts
0,12,2,<removed>,<removed>,1,0,0,0,autosubmit,0,deferredfeedback,0,0,0,1,2,-1,69888,4352,4352,4352,4352,4352,4352,1,free,1,2.0,10.0,0,1570013984,NaN,NaN,-,0,0,0,0,0,0,0
1,55,2,<removed>,<removed>,1,0,0,0,autosubmit,0,deferredfeedback,0,0,0,1,2,-1,69888,4352,4352,4352,4352,0,4352,1,free,1,12.0,10.0,0,1574288876,NaN,NaN,-,0,0,0,0,0,0,0
2,20,2,<removed>,<removed>,1,0,0,0,autosubmit,0,deferredfeedback,0,0,0,1,2,-1,69888,4352,4352,4352,4352,0,4352,1,free,1,12.0,10.0,0,1574774064,NaN,NaN,-,0,0,0,0,0,0,0
3,4,3,<removed>,<removed>,1,0,0,0,autoabandon,0,deferredfeedback,0,0,0,1,2,-1,69888,4352,4352,4352,4352,4352,4352,1,free,1,40.0,100.0,0,1580831338,NaN,NaN,-,0,0,0,0,0,1,0
4,7,2,<removed>,<removed>,1,0,0,0,autosubmit,0,deferredfeedback,0,0,0,1,2,-1,69888,4352,4352,4352,4352,4352,4352,1,free,1,2.0,10.0,0,1570015806,NaN,NaN,-,0,0,0,0,0,0,0


In [169]:
m_assign.columns.tolist()

['id',
 'course',
 'name',
 'intro',
 'introformat',
 'alwaysshowdescription',
 'nosubmissions',
 'submissiondrafts',
 'sendnotifications',
 'sendlatenotifications',
 'duedate',
 'allowsubmissionsfromdate',
 'grade',
 'timemodified',
 'requiresubmissionstatement',
 'completionsubmit',
 'cutoffdate',
 'gradingduedate',
 'teamsubmission',
 'requireallteammemberssubmit',
 'teamsubmissiongroupingid',
 'blindmarking',
 'revealidentities',
 'attemptreopenmethod',
 'maxattempts',
 'markingworkflow',
 'markingallocation',
 'sendstudentnotifications',
 'preventsubmissionnotingroup']

In [170]:
m_assign.columns.tolist()

['id',
 'course',
 'name',
 'intro',
 'introformat',
 'alwaysshowdescription',
 'nosubmissions',
 'submissiondrafts',
 'sendnotifications',
 'sendlatenotifications',
 'duedate',
 'allowsubmissionsfromdate',
 'grade',
 'timemodified',
 'requiresubmissionstatement',
 'completionsubmit',
 'cutoffdate',
 'gradingduedate',
 'teamsubmission',
 'requireallteammemberssubmit',
 'teamsubmissiongroupingid',
 'blindmarking',
 'revealidentities',
 'attemptreopenmethod',
 'maxattempts',
 'markingworkflow',
 'markingallocation',
 'sendstudentnotifications',
 'preventsubmissionnotingroup']

In [155]:
csv_05 = list(
    repo_05.rglob("*.csv")
)

for archivo in csv_05:

    print("\n" + "=" * 100)
    print(archivo.name)
    print("=" * 100)

    df = pd.read_csv(
        archivo,
        nrows=1000,
        low_memory=False
    )

    print("Dimensiones muestra:", df.shape)

    display(df.head())

    print("\nVariables:")
    print(df.columns.tolist())


m_assign.csv
Dimensiones muestra: (68, 1)


,id;course;name;intro;introformat;alwaysshowdescription;nosubmissions;submissiondrafts;sendnotifications;sendlatenotifications;duedate;allowsubmissionsfromdate;grade;timemodified;requiresubmissionstatement;completionsubmit;cutoffdate;gradingduedate;teamsubmission;requireallteammemberssubmit;teamsubmissiongroupingid;blindmarking;revealidentities;attemptreopenmethod;maxattempts;markingworkflow;markingallocation;sendstudentnotifications;preventsubmissionnotingroup
0,4;2;Einsendeaufgabe 1.1;<removed>;1;1;0;0;0;0;1572303540;1569276000;6;1571305062;0;1;1572303540;0;0;0;0;0;0;none;-1;0;0;1;0
1,20;2;Einsendeaufgabe 2.8;<removed>;1;1;0;0;0;0;1574722740;1569276000;8;1569400539;0;1;1574722740;0;0;0;0;0;0;none;-1;0;0;1;0
2,41;5;Einsendeaufgabe 1.2;<removed>;1;1;0;0;0;0;1603666740;1600984800;5;1599809892;0;1;1603753140;0;0;0;0;0;0;none;-1;0;0;1;0
3,30;2;Einsendeaufgabe 3.7;<removed>;1;1;0;0;0;0;1578351540;1569362400;10;1569400782;0;1;1578351540;0;0;0;0;0;0;none;-1;0;0;1;0
4,11;2;Einsendeaufgabe 1.8;<removed>;1;1;0;0;0;0;1572303540;1569276000;14;1569400337;0;1;1572303540;0;0;0;0;0;0;none;-1;0;0;1;0



Variables:
['id;course;name;intro;introformat;alwaysshowdescription;nosubmissions;submissiondrafts;sendnotifications;sendlatenotifications;duedate;allowsubmissionsfromdate;grade;timemodified;requiresubmissionstatement;completionsubmit;cutoffdate;gradingduedate;teamsubmission;requireallteammemberssubmit;teamsubmissiongroupingid;blindmarking;revealidentities;attemptreopenmethod;maxattempts;markingworkflow;markingallocation;sendstudentnotifications;preventsubmissionnotingroup']

m_quiz.csv
Dimensiones muestra: (158, 1)


,id;course;name;intro;introformat;timeopen;timeclose;timelimit;overduehandling;graceperiod;preferredbehaviour;canredoquestions;attempts;attemptonlast;grademethod;decimalpoints;questiondecimalpoints;reviewattempt;reviewcorrectness;reviewmarks;reviewspecificfeedback;reviewgeneralfeedback;reviewrightanswer;reviewoverallfeedback;questionsperpage;navmethod;shuffleanswers;sumgrades;grade;timecreated;timemodified;password;subnet;browsersecurity;delay1;delay2;showuserpicture;showblocks;completionattemptsexhausted;completionpass;allowofflineattempts
0,12;2;<removed>;<removed>;1;0;0;0;autosubmit;0;deferredfeedback;0;0;0;1;2;-1;69888;4352;4352;4352;4352;4352;4352;1;free;1;2.0;10.0;0;1570013984;;;-...
1,55;2;<removed>;<removed>;1;0;0;0;autosubmit;0;deferredfeedback;0;0;0;1;2;-1;69888;4352;4352;4352;4352;0;4352;1;free;1;12.0;10.0;0;1574288876;;;-;0...
2,20;2;<removed>;<removed>;1;0;0;0;autosubmit;0;deferredfeedback;0;0;0;1;2;-1;69888;4352;4352;4352;4352;0;4352;1;free;1;12.0;10.0;0;1574774064;;;-;0...
3,4;3;<removed>;<removed>;1;0;0;0;autoabandon;0;deferredfeedback;0;0;0;1;2;-1;69888;4352;4352;4352;4352;4352;4352;1;free;1;40.0;100.0;0;1580831338;;...
4,7;2;<removed>;<removed>;1;0;0;0;autosubmit;0;deferredfeedback;0;0;0;1;2;-1;69888;4352;4352;4352;4352;4352;4352;1;free;1;2.0;10.0;0;1570015806;;;-;...



Variables:
['id;course;name;intro;introformat;timeopen;timeclose;timelimit;overduehandling;graceperiod;preferredbehaviour;canredoquestions;attempts;attemptonlast;grademethod;decimalpoints;questiondecimalpoints;reviewattempt;reviewcorrectness;reviewmarks;reviewspecificfeedback;reviewgeneralfeedback;reviewrightanswer;reviewoverallfeedback;questionsperpage;navmethod;shuffleanswers;sumgrades;grade;timecreated;timemodified;password;subnet;browsersecurity;delay1;delay2;showuserpicture;showblocks;completionattemptsexhausted;completionpass;allowofflineattempts']


In [156]:
def contar_unicos_seguro(serie):
    """
    Cuenta valores únicos incluso cuando la columna
    contiene listas o diccionarios.
    """

    serie = serie.dropna()

    if serie.empty:
        return 0

    def convertir_hashable(valor):

        if isinstance(valor, (list, dict)):
            return json.dumps(
                valor,
                sort_keys=True,
                ensure_ascii=False
            )

        if isinstance(valor, set):
            return tuple(
                sorted(valor)
            )

        return valor

    try:

        return (
            serie
            .map(convertir_hashable)
            .nunique()
        )

    except (TypeError, ValueError):

        return (
            serie
            .astype(str)
            .nunique()
        )


def resumen_dataframe(
    df,
    nombre="DataFrame"
):

    print("=" * 100)
    print(nombre)
    print("=" * 100)

    print(
        f"Filas: {df.shape[0]:,}"
    )

    print(
        f"Columnas: {df.shape[1]:,}"
    )

    resumen = pd.DataFrame({

        "variable":
            df.columns,

        "tipo":
            df.dtypes.astype(str).values,

        "nulos":
            df.isna().sum().values,

        "porcentaje_nulos":
            (
                df.isna().mean().values * 100
            ).round(2),

        "valores_unicos": [
            contar_unicos_seguro(
                df[col]
            )
            for col in df.columns
        ]

    })

    return resumen

In [157]:
resumen_dataframe(
    aggregates_sample,
    "Dataset 28489601 - Aggregates"
)

Dataset 28489601 - Aggregates
Filas: 1,000
Columnas: 1,235


,variable,tipo,nulos,porcentaje_nulos,valores_unicos
0,middle.id,str,0,0.0,1000
1,middle.user_id,str,0,0.0,1
2,middle.timestamp,int64,0,0.0,996
3,middle.url,str,0,0.0,18
4,middle.mouse.position.x,int64,0,0.0,116
...,...,...,...,...,...
1230,2000.after.speed.x.avg,float64,0,0.0,93
1231,2000.after.speed.x.std,float64,0,0.0,93
1232,2000.after.speed.y.sum,float64,0,0.0,87
1233,2000.after.speed.y.avg,float64,0,0.0,94


In [158]:
resumen_dataframe(
    interactions_sample,
    "Dataset 28489601 - Interactions"
)

Dataset 28489601 - Interactions
Filas: 1,000
Columnas: 38


,variable,tipo,nulos,porcentaje_nulos,valores_unicos
0,id,str,0,0.0,1000
1,user_id,str,0,0.0,1
2,timestamp,int64,0,0.0,996
3,url,str,0,0.0,3
4,mouse.position.x,int64,0,0.0,269
5,mouse.position.y,int64,0,0.0,264
6,mouse.clicks,bool,0,0.0,2
7,mouse.clicks.left,bool,0,0.0,2
8,mouse.clicks.right,bool,0,0.0,1
9,mouse.clicks.middle,bool,0,0.0,1


In [159]:
resumen_dataframe(
    a11y_metadata,
    "Reduced A11y-CUA - Metadata"
)

Reduced A11y-CUA - Metadata
Filas: 1,320
Columnas: 38


,variable,tipo,nulos,porcentaje_nulos,valores_unicos
0,task.task_id,int64,0,0.00,60
1,task.task_title,str,0,0.00,64
2,task.instruction,str,0,0.00,73
3,task.context,str,0,0.00,64
4,task.task_group,str,0,0.00,7
5,task.familiarity,str,0,0.00,3
6,task.difficulty_before,str,0,0.00,4
7,task.difficulty_after,str,12,0.91,4
8,task.success,bool,0,0.00,2
9,session.started_at,float64,0,0.00,1320


In [160]:
resumen_dataframe(
    prototype_metadata,
    "29042675 - Prototype Metadata"
)

29042675 - Prototype Metadata
Filas: 20
Columnas: 4


,variable,tipo,nulos,porcentaje_nulos,valores_unicos
0,Prototype,str,0,0.0,19
1,Prototype Type,str,0,0.0,2
2,Domain,str,0,0.0,12
3,Scource URL,str,0,0.0,20


In [161]:
dataframes = {
    "28489601_aggregates": aggregates_sample,
    "28489601_interactions": interactions_sample,
    "A11y_CUA_metadata": a11y_metadata,
    "A11y_CUA_a11y_sample": a11y_tree_sample,
    "A11y_CUA_web_log_sample": web_log_sample,
    "29042675_prototype_metadata": prototype_metadata
}

In [162]:
resumen_general = []

for nombre, df in dataframes.items():

    resumen_general.append({
        "dataset": nombre,
        "filas_cargadas": df.shape[0],
        "columnas": df.shape[1],
        "memoria_mb": (
            df.memory_usage(
                deep=True
            ).sum()
            / 1024**2
        )
    })

pd.DataFrame(
    resumen_general
)

,dataset,filas_cargadas,columnas,memoria_mb
0,28489601_aggregates,1000,1235,9.530408
1,28489601_interactions,1000,38,0.396935
2,A11y_CUA_metadata,1320,38,2.490954
3,A11y_CUA_a11y_sample,1,8,0.000448
4,A11y_CUA_web_log_sample,1,9,0.000773
5,29042675_prototype_metadata,20,4,0.006371


In [163]:
aggregates_sample.columns.tolist()

['middle.id',
 'middle.user_id',
 'middle.timestamp',
 'middle.url',
 'middle.mouse.position.x',
 'middle.mouse.position.y',
 'middle.mouse.clicks',
 'middle.mouse.clicks.left',
 'middle.mouse.clicks.right',
 'middle.mouse.clicks.middle',
 'middle.mouse.clicks.others',
 'middle.mouse.speed',
 'middle.mouse.speed.x',
 'middle.mouse.speed.y',
 'middle.mouse.acceleration',
 'middle.mouse.acceleration.x',
 'middle.mouse.acceleration.y',
 'middle.trajectory.slope',
 'middle.scroll.absolute.x',
 'middle.scroll.absolute.y',
 'middle.scroll.relative.x',
 'middle.scroll.relative.y',
 'middle.keyboard',
 'middle.keyboard.alpha',
 'middle.keyboard.numeric',
 'middle.keyboard.function',
 'middle.keyboard.symbol',
 'middle.emotions.automatic_sampling',
 'middle.emotions.exist',
 'middle.emotions.joy',
 'middle.emotions.fear',
 'middle.emotions.disgust',
 'middle.emotions.sadness',
 'middle.emotions.anger',
 'middle.emotions.surprise',
 'middle.emotions.contempt',
 'middle.emotions.valence',
 'middl

In [164]:
interactions_sample.columns.tolist()

['id',
 'user_id',
 'timestamp',
 'url',
 'mouse.position.x',
 'mouse.position.y',
 'mouse.clicks',
 'mouse.clicks.left',
 'mouse.clicks.right',
 'mouse.clicks.middle',
 'mouse.clicks.others',
 'mouse.speed',
 'mouse.speed.x',
 'mouse.speed.y',
 'mouse.acceleration',
 'mouse.acceleration.x',
 'mouse.acceleration.y',
 'trajectory.slope',
 'scroll.absolute.x',
 'scroll.absolute.y',
 'scroll.relative.x',
 'scroll.relative.y',
 'keyboard',
 'keyboard.alpha',
 'keyboard.numeric',
 'keyboard.function',
 'keyboard.symbol',
 'emotions.automatic_sampling',
 'emotions.exist',
 'emotions.joy',
 'emotions.fear',
 'emotions.disgust',
 'emotions.sadness',
 'emotions.anger',
 'emotions.surprise',
 'emotions.contempt',
 'emotions.valence',
 'emotions.engagement']

In [165]:
a11y_metadata.columns.tolist()

['task.task_id',
 'task.task_title',
 'task.instruction',
 'task.context',
 'task.task_group',
 'task.familiarity',
 'task.difficulty_before',
 'task.difficulty_after',
 'task.success',
 'session.started_at',
 'session.ended_at',
 'session.folder',
 'session.applications',
 'accessibility.baseline.os',
 'accessibility.baseline.os_version',
 'accessibility.baseline.python_version',
 'accessibility.baseline.narrator.enabled',
 'accessibility.baseline.narrator.startup_enabled',
 'accessibility.baseline.magnifier.enabled',
 'accessibility.baseline.magnifier.zoom',
 'accessibility.baseline.color_filter.enabled',
 'accessibility.baseline.color_filter.type',
 'accessibility.baseline.high_contrast.enabled',
 'accessibility.baseline.high_contrast.flags',
 'accessibility.baseline.sticky_keys.flags',
 'accessibility.baseline.sticky_keys.maybe_enabled',
 'accessibility.baseline.toggle_keys.flags',
 'accessibility.baseline.toggle_keys.maybe_enabled',
 'accessibility.baseline.filter_keys.flags',
 'a

In [166]:
web_log_sample.columns.tolist()

['url',
 'title',
 'order',
 'created_at',
 'dom_file',
 'dom_url',
 'a11y_file',
 'a11y_url',
 'interactions']

In [167]:
prototype_metadata.columns.tolist()

['Prototype', 'Prototype Type', 'Domain', 'Scource URL']

## DATA PREPARATION

In [171]:
INTERIM_DIR = Path("../data/interim").resolve()

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(INTERIM_DIR)

/Users/uriel/Documents/machine_learning/data/interim


A11y-CUA normalizado y enviado a interim

In [173]:
a11y_instances = pd.DataFrame()

# Identificador de la carpeta/sesión
a11y_instances["session_dir"] = (
    a11y_metadata["source_file"]
    .map(lambda x: str(Path(x).parent))
)

# Identificador único de instancia
a11y_instances["instance_id"] = (
    a11y_instances["session_dir"]
    .str.replace(r"[^A-Za-z0-9]+", "_", regex=True)
)

# Información de tarea
a11y_instances["task_id"] = (
    a11y_metadata["task.task_id"]
)

a11y_instances["task_title"] = (
    a11y_metadata["task.task_title"]
)

a11y_instances["task_type"] = (
    a11y_metadata["task.task_group"]
)

a11y_instances["task_context"] = (
    a11y_metadata["task.context"]
)

# Resultado
a11y_instances["task_completed"] = (
    a11y_metadata["task.success"]
    .astype("Int8")
)

# Timestamps
a11y_instances["started_at"] = (
    a11y_metadata["session.started_at"]
)

a11y_instances["ended_at"] = (
    a11y_metadata["session.ended_at"]
)

# Tiempo real de la tarea
a11y_instances["task_time_s"] = (
    a11y_instances["ended_at"]
    - a11y_instances["started_at"]
)

# Grupo original del dataset
a11y_instances["source_group"] = (
    a11y_metadata["grupo"]
)

a11y_instances.head()

,session_dir,instance_id,task_id,task_title,task_type,task_context,task_completed,started_at,ended_at,task_time_s,source_group
0,Qwen3-VL-32B-Instruct/Screen Reader/59,Qwen3_VL_32B_Instruct_Screen_Reader_59,59,Enable Picture-in-Picture playback,Media,I want a video I am watching on YouTube to float in a small window while I look for other videos.,0,1.764911e+09,1.764912e+09,723.138184,Qwen3-VL-32B-Instruct
1,Qwen3-VL-32B-Instruct/Screen Reader/50,Qwen3_VL_32B_Instruct_Screen_Reader_50,50,Change the video speed,Media,"I'm watching a video on Media Player, but I'm on a time crunch so I want to speed through it.",0,1.764907e+09,1.764908e+09,731.839376,Qwen3-VL-32B-Instruct
2,Qwen3-VL-32B-Instruct/Screen Reader/57,Qwen3_VL_32B_Instruct_Screen_Reader_57,57,Change the language of the video,Media,"I'm watching a YouTube video, but this video is in English and I want to improve my Spanish ability.",0,1.764908e+09,NaN,NaN,Qwen3-VL-32B-Instruct
3,Qwen3-VL-32B-Instruct/Screen Reader/32,Qwen3_VL_32B_Instruct_Screen_Reader_32,32,Add a strike-through and highlight to text in Word,Document Editing,I'm reading a word document. I want to suggest changes to the article's title by striking out the title from the document so that the author is aw...,0,1.764898e+09,1.764899e+09,574.484475,Qwen3-VL-32B-Instruct
4,Qwen3-VL-32B-Instruct/Screen Reader/35,Qwen3_VL_32B_Instruct_Screen_Reader_35,35,Capitalize the starting letter of every word in Word and add page number,Document Editing,I am writing an article on Word and I want to modify the title format based on the guidelines.,0,1.764902e+09,1.764903e+09,539.837911,Qwen3-VL-32B-Instruct


In [174]:
a11y_instances.shape

(1320, 11)

In [175]:
a11y_instances["instance_id"].duplicated().sum()

np.int64(0)

In [176]:
a11y_instances[
    [
        "instance_id",
        "task_type",
        "task_completed",
        "task_time_s"
    ]
].head(20)

,instance_id,task_type,task_completed,task_time_s
0,Qwen3_VL_32B_Instruct_Screen_Reader_59,Media,0,723.138184
1,Qwen3_VL_32B_Instruct_Screen_Reader_50,Media,0,731.839376
2,Qwen3_VL_32B_Instruct_Screen_Reader_57,Media,0,NaN
3,Qwen3_VL_32B_Instruct_Screen_Reader_32,Document Editing,0,574.484475
4,Qwen3_VL_32B_Instruct_Screen_Reader_35,Document Editing,0,539.837911
5,Qwen3_VL_32B_Instruct_Screen_Reader_56,Media,0,NaN
6,Qwen3_VL_32B_Instruct_Screen_Reader_51,Media,0,741.140007
7,Qwen3_VL_32B_Instruct_Screen_Reader_58,Media,0,678.711941
8,Qwen3_VL_32B_Instruct_Screen_Reader_60,Media,0,744.375465
9,Qwen3_VL_32B_Instruct_Screen_Reader_34,Document Editing,0,554.000624


In [196]:
a11y_instances.to_csv(
    INTERIM_DIR / "a11y_instances.csv",
    index=False
)

print(
    "Guardado:",
    INTERIM_DIR / "a11y_instances.csv"
)

Guardado: /Users/uriel/Documents/machine_learning/data/interim/a11y_instances.csv


In [197]:
a11y_web_events.to_csv(
    INTERIM_DIR / "a11y_web_events.csv",
    index=False
)

print(
    "Guardado:",
    INTERIM_DIR / "a11y_web_events.csv"
)

print(
    "Dimensiones:",
    a11y_web_events.shape
)

Guardado: /Users/uriel/Documents/machine_learning/data/interim/a11y_web_events.csv
Dimensiones: (28477, 25)


In [178]:
a11y_instances["source_type"] = np.where(
    a11y_instances["source_group"]
    .str.contains(
        "Claude|Qwen",
        case=False,
        na=False
    ),
    "agent",
    "human_candidate"
)

In [179]:
a11y_instances[
    ["source_group", "source_type"]
].drop_duplicates()

,source_group,source_type
0,Qwen3-VL-32B-Instruct,agent
180,BLVU,human_candidate
660,SU,human_candidate
1140,Claude-Sonnet-4.5,agent


In [180]:
def cargar_interacciones_web_a11y(
    archivos,
    raiz
):
    registros = []
    errores = []

    for archivo in archivos:

        try:

            with open(
                archivo,
                "r",
                encoding="utf-8"
            ) as f:
                data = json.load(f)

            interactions = data.get(
                "interactions",
                []
            )

            if not isinstance(
                interactions,
                list
            ):
                continue

            if len(interactions) == 0:
                continue

            eventos = pd.json_normalize(
                interactions,
                sep="."
            )

            # Carpeta correspondiente a la tarea
            session_dir = str(
                archivo
                .parent
                .parent
                .relative_to(raiz)
            )

            eventos["session_dir"] = session_dir

            eventos["page_url"] = data.get(
                "url"
            )

            eventos["page_title"] = data.get(
                "title"
            )

            eventos["page_order"] = data.get(
                "order"
            )

            eventos["page_created_at"] = data.get(
                "created_at"
            )

            eventos["source_log"] = str(
                archivo.relative_to(raiz)
            )

            registros.append(
                eventos
            )

        except Exception as e:

            errores.append({
                "archivo": str(archivo),
                "error": str(e)
            })

    if not registros:
        return (
            pd.DataFrame(),
            pd.DataFrame(errores)
        )

    return (
        pd.concat(
            registros,
            ignore_index=True
        ),
        pd.DataFrame(errores)
    )

In [181]:
a11y_web_events, errores_web = (
    cargar_interacciones_web_a11y(
        web_log_json_files,
        a11y_root
    )
)

In [182]:
a11y_web_events.shape

(28477, 25)

In [183]:
a11y_web_events.head()

,type,timestamp,selector,element.tag,element.id,element.class,element.role,element.ariaLabel,element.name,element.href,element.text,session_dir,page_url,page_title,page_order,page_created_at,source_log,element.value,x,y,key,scrollTop,scrollLeft,windowHeight,documentHeight
0,focus,1764844377282,a.globalnav-link.globalnav-submenu-trigger-link.globalnav-link-store,A,,globalnav-link globalnav-submenu-trigger-link globalnav-link-store,None,Store,None,/us/shop/goto/store,Store,Qwen3-VL-32B-Instruct/Default/3,https://www.apple.com/,Apple,1,1764844377283,Qwen3-VL-32B-Instruct/Default/3/web_logs/web_tab1_1764844377283.json,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,blur,1764845235522,button.ytSearchboxComponentSearchButton,BUTTON,,ytSearchboxComponentSearchButton,None,Search,None,NaN,,Qwen3-VL-32B-Instruct/Default/8,https://www.youtube.com/results?search_query=Machine+Learning+and+Generative+AI,Machine Learning and Generative AI - YouTube,2,1764845235523,Qwen3-VL-32B-Instruct/Default/8/web_logs/web_tab2_1764845235523.json,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,focus,1764845235523,#button,BUTTON,button,style-scope yt-icon-button,None,Guide,None,NaN,,Qwen3-VL-32B-Instruct/Default/8,https://www.youtube.com/results?search_query=Machine+Learning+and+Generative+AI,Machine Learning and Generative AI - YouTube,2,1764845235523,Qwen3-VL-32B-Instruct/Default/8/web_logs/web_tab2_1764845235523.json,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,blur,1764845236751,#button,BUTTON,button,style-scope yt-icon-button,None,Guide,None,NaN,,Qwen3-VL-32B-Instruct/Default/8,https://www.youtube.com/results?search_query=Machine+Learning+and+Generative+AI,Machine Learning and Generative AI - YouTube,2,1764845235523,Qwen3-VL-32B-Instruct/Default/8/web_logs/web_tab2_1764845235523.json,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,focus,1764845236753,button.ytSearchboxComponentSearchButton,BUTTON,,ytSearchboxComponentSearchButton,None,Search,None,NaN,,Qwen3-VL-32B-Instruct/Default/8,https://www.youtube.com/results?search_query=Machine+Learning+and+Generative+AI,Machine Learning and Generative AI - YouTube,2,1764845235523,Qwen3-VL-32B-Instruct/Default/8/web_logs/web_tab2_1764845235523.json,,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [184]:
a11y_web_events.columns.tolist()

['type',
 'timestamp',
 'selector',
 'element.tag',
 'element.id',
 'element.class',
 'element.role',
 'element.ariaLabel',
 'element.name',
 'element.href',
 'element.text',
 'session_dir',
 'page_url',
 'page_title',
 'page_order',
 'page_created_at',
 'source_log',
 'element.value',
 'x',
 'y',
 'key',
 'scrollTop',
 'scrollLeft',
 'windowHeight',
 'documentHeight']

In [185]:
a11y_web_events["type"].value_counts(
    dropna=False
)

type
key_press      8394
focus          5107
blur           5069
input          4519
scroll         3677
mouse_click    1088
page            623
Name: count, dtype: int64

In [186]:
def aplanar_arbol_a11y(
    nodo,
    profundidad=0,
    parent_id=None,
    filas=None
):

    if filas is None:
        filas = []

    node_id = len(filas)

    rect = nodo.get(
        "bounding_rectangle",
        {}
    ) or {}

    filas.append({
        "node_id": node_id,
        "parent_id": parent_id,
        "depth": profundidad,

        "name": nodo.get(
            "name"
        ),

        "control_type": nodo.get(
            "control_type"
        ),

        "automation_id": nodo.get(
            "automation_id"
        ),

        "left": rect.get(
            "left"
        ),

        "top": rect.get(
            "top"
        ),

        "right": rect.get(
            "right"
        ),

        "bottom": rect.get(
            "bottom"
        )
    })

    children = nodo.get(
        "children",
        []
    ) or []

    for child in children:

        if isinstance(child, dict):

            aplanar_arbol_a11y(
                child,
                profundidad + 1,
                node_id,
                filas
            )

    return filas

In [187]:
nodes = aplanar_arbol_a11y(
    a11y_example
)

a11y_nodes_sample = pd.DataFrame(
    nodes
)

a11y_nodes_sample.head(30)

,node_id,parent_id,depth,name,control_type,automation_id,left,top,right,bottom
0,0,NaN,0,readme.md - computer-use-agents - Visual Studio Code,PaneControl,,93,0,1863,1008
1,1,0.0,1,readme.md - computer-use-agents - Visual Studio Code,DocumentControl,841024,93,0,1863,1008
2,2,1.0,2,,GroupControl,,93,0,1863,1008
3,3,2.0,3,,WindowControl,,93,0,1863,1008
4,4,3.0,4,,GroupControl,,93,0,1863,1008
5,5,3.0,4,,GroupControl,,0,0,0,0
6,6,0.0,1,,PaneControl,,93,0,1863,1008
7,7,0.0,1,,PaneControl,,93,0,1863,1008
8,8,7.0,2,,PaneControl,,93,0,1863,1008
9,9,8.0,3,,PaneControl,,1657,1,1862,51


In [188]:
a11y_nodes_sample.shape

(15, 10)

In [189]:
a11y_nodes_sample[
    "control_type"
].value_counts()

control_type
PaneControl        7
GroupControl       3
ButtonControl      3
DocumentControl    1
WindowControl      1
Name: count, dtype: int64

In [190]:
m_assign.shape, m_quiz.shape

((68, 29), (158, 41))

In [191]:
a11y_instances.head()

,session_dir,instance_id,task_id,task_title,task_type,task_context,task_completed,started_at,ended_at,task_time_s,source_group,source_type
0,Qwen3-VL-32B-Instruct/Screen Reader/59,Qwen3_VL_32B_Instruct_Screen_Reader_59,59,Enable Picture-in-Picture playback,Media,I want a video I am watching on YouTube to float in a small window while I look for other videos.,0,1.764911e+09,1.764912e+09,723.138184,Qwen3-VL-32B-Instruct,agent
1,Qwen3-VL-32B-Instruct/Screen Reader/50,Qwen3_VL_32B_Instruct_Screen_Reader_50,50,Change the video speed,Media,"I'm watching a video on Media Player, but I'm on a time crunch so I want to speed through it.",0,1.764907e+09,1.764908e+09,731.839376,Qwen3-VL-32B-Instruct,agent
2,Qwen3-VL-32B-Instruct/Screen Reader/57,Qwen3_VL_32B_Instruct_Screen_Reader_57,57,Change the language of the video,Media,"I'm watching a YouTube video, but this video is in English and I want to improve my Spanish ability.",0,1.764908e+09,NaN,NaN,Qwen3-VL-32B-Instruct,agent
3,Qwen3-VL-32B-Instruct/Screen Reader/32,Qwen3_VL_32B_Instruct_Screen_Reader_32,32,Add a strike-through and highlight to text in Word,Document Editing,I'm reading a word document. I want to suggest changes to the article's title by striking out the title from the document so that the author is aw...,0,1.764898e+09,1.764899e+09,574.484475,Qwen3-VL-32B-Instruct,agent
4,Qwen3-VL-32B-Instruct/Screen Reader/35,Qwen3_VL_32B_Instruct_Screen_Reader_35,35,Capitalize the starting letter of every word in Word and add page number,Document Editing,I am writing an article on Word and I want to modify the title format based on the guidelines.,0,1.764902e+09,1.764903e+09,539.837911,Qwen3-VL-32B-Instruct,agent


In [192]:
a11y_web_events["type"].value_counts(dropna=False)

type
key_press      8394
focus          5107
blur           5069
input          4519
scroll         3677
mouse_click    1088
page            623
Name: count, dtype: int64

In [193]:
a11y_nodes_sample["control_type"].value_counts()

control_type
PaneControl        7
GroupControl       3
ButtonControl      3
DocumentControl    1
WindowControl      1
Name: count, dtype: int64

In [194]:
irr = pd.read_csv(
    RAW_DIR / "03" / "29042675" / "IRR_Results.csv"
)

print(irr.shape)

print(
    irr.columns.tolist()
)

display(
    irr.head(10)
)

(104, 36)
['Name', 'Prototype Link', 'H1 Compare', 'H2 Compare', 'H3 Compare', 'H4 Compare', 'H5 Compare', 'H6 Compare', 'H7 Compare', 'H8 Compare', 'H9 Compare', 'H10 Compare', 'Unnamed: 12', 'Unnamed: 13', 'H1. Overal Rating (4)', 'H1', 'H2. Overal Rating(4)', 'H2', 'H3. Overal Rating(4)', 'H3', 'H4 Overall Rating', 'H4', 'H5 Overall Rating', 'H5', 'H6 Overall Rating', 'H6', 'H7 Overall Rating', 'H7', 'H8 Overall Rating', 'H8', 'H9 Overall Rating', 'H9', 'H10 Overall Rating', 'H10', 'Unnamed: 0', 'Unnamed: 11']


,Name,Prototype Link,H1 Compare,H2 Compare,H3 Compare,H4 Compare,H5 Compare,H6 Compare,H7 Compare,H8 Compare,H9 Compare,H10 Compare,Unnamed: 12,Unnamed: 13,H1. Overal Rating (4),H1,H2. Overal Rating(4),H2,H3. Overal Rating(4),H3,H4 Overall Rating,H4,H5 Overall Rating,H5,H6 Overall Rating,H6,H7 Overall Rating,H7,H8 Overall Rating,H8,H9 Overall Rating,H9,H10 Overall Rating,H10,Unnamed: 0,Unnamed: 11
0,Food Order,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App-(Community)?node-id=0-1&p=f&t=Dx8Q70dOPPnSAAPO-0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gaira (Community app),https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E-Tutor,https://www.figma.com/community/file/1271304360859402063,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tracking App,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Travel,https://www.figma.com/community/file/1379689864793179441,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Running,https://www.behance.net/gallery/158363369/UI-Projects/modules/893495869,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Bakery,https://www.figma.com/community/file/988854619809579450,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Hospital 1,https://www.figma.com/community/file/1250314760992299521,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Hospital 2,https://www.figma.com/community/file/12725823068122925,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Housing,https://www.figma.com/design/QLcZcyPAov1fsHDEfnX6T4/Real-Estate-Agency---Landing-Page-Design-(Community)?node-id=0-1&p=f&t=IBZxitN7XpPIezha-0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [198]:
INTERACTIVE_CONTROLS = {
    "ButtonControl",
    "EditControl",
    "ComboBoxControl",
    "CheckBoxControl",
    "RadioButtonControl",
    "HyperlinkControl",
    "MenuItemControl",
    "TabItemControl",
    "ListItemControl",
    "SliderControl",
    "SpinnerControl"
}

In [199]:
def extraer_nodos_a11y(
    nodo,
    depth=0,
    filas=None
):

    if filas is None:
        filas = []

    filas.append({
        "name": nodo.get("name"),
        "control_type": nodo.get("control_type"),
        "automation_id": nodo.get("automation_id"),
        "depth": depth
    })

    children = nodo.get(
        "children",
        []
    ) or []

    for child in children:

        if isinstance(
            child,
            dict
        ):

            extraer_nodos_a11y(
                child,
                depth + 1,
                filas
            )

    return filas

In [200]:
def resumir_arbol_a11y(
    archivo,
    raiz
):

    with open(
        archivo,
        "r",
        encoding="utf-8"
    ) as f:

        tree = json.load(f)

    nodos = extraer_nodos_a11y(
        tree
    )

    df_nodes = pd.DataFrame(
        nodos
    )

    interactive = df_nodes[
        df_nodes["control_type"]
        .isin(INTERACTIVE_CONTROLS)
    ].copy()

    nombres_validos = (
        interactive["name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    interactive_count = len(
        interactive
    )

    named_count = int(
        nombres_validos.sum()
    )

    unlabeled_count = (
        interactive_count
        - named_count
    )

    if interactive_count > 0:

        accessible_name_ratio = (
            named_count
            / interactive_count
        )

    else:

        accessible_name_ratio = np.nan

    relativa = (
        archivo
        .relative_to(raiz)
    )

    if archivo.parent.name == "web_logs":

        session_folder = (
            archivo.parent.parent
        )

    else:

        session_folder = (
            archivo.parent
        )

    session_dir = str(
        session_folder
        .relative_to(raiz)
    )

    return {
        "session_dir": session_dir,
        "source_file": str(relativa),

        "node_count": len(
            df_nodes
        ),

        "interactive_controls_count":
            interactive_count,

        "named_interactive_controls":
            named_count,

        "unlabeled_controls_count":
            unlabeled_count,

        "accessible_name_ratio":
            accessible_name_ratio,

        "max_a11y_tree_depth":
            df_nodes["depth"].max()
            if len(df_nodes)
            else np.nan
    }

In [201]:
a11y_tree_records = []

errores_a11y = []

for i, archivo in enumerate(
    a11y_tree_files,
    start=1
):

    try:

        resultado = resumir_arbol_a11y(
            archivo,
            a11y_root
        )

        a11y_tree_records.append(
            resultado
        )

    except Exception as e:

        errores_a11y.append({
            "archivo": str(archivo),
            "error": str(e)
        })

    if i % 500 == 0:

        print(
            f"{i:,}/"
            f"{len(a11y_tree_files):,}"
        )

500/7,419
1,000/7,419
1,500/7,419
2,000/7,419
2,500/7,419
3,000/7,419
3,500/7,419
4,000/7,419
4,500/7,419
5,000/7,419
5,500/7,419
6,000/7,419
6,500/7,419
7,000/7,419


In [202]:
a11y_tree_features = pd.DataFrame(
    a11y_tree_records
)

a11y_tree_features.shape

(7368, 8)

In [203]:
a11y_tree_features.head()

,session_dir,source_file,node_count,interactive_controls_count,named_interactive_controls,unlabeled_controls_count,accessible_name_ratio,max_a11y_tree_depth
0,Qwen3-VL-32B-Instruct/Screen Reader/59,Qwen3-VL-32B-Instruct/Screen Reader/59/1_Code.exe_1764911116433_a11y_tree.json,15,3,3,0,1.0,4
1,Qwen3-VL-32B-Instruct/Screen Reader/32,Qwen3-VL-32B-Instruct/Screen Reader/32/1_Code.exe_1764898115564_a11y_tree.json,15,3,3,0,1.0,4
2,Qwen3-VL-32B-Instruct/Screen Reader/58,Qwen3-VL-32B-Instruct/Screen Reader/58/1_Code.exe_1764908208518_a11y_tree.json,15,3,3,0,1.0,4
3,Qwen3-VL-32B-Instruct/Screen Reader/34,Qwen3-VL-32B-Instruct/Screen Reader/34/1_Code.exe_1764903414601_a11y_tree.json,15,3,3,0,1.0,4
4,Qwen3-VL-32B-Instruct/Screen Reader/33,Qwen3-VL-32B-Instruct/Screen Reader/33/1_Code.exe_1764898698046_a11y_tree.json,15,3,3,0,1.0,4


In [204]:
a11y_tree_features.describe().T

,count,mean,std,min,25%,50%,75%,max
node_count,7368.0,266.715798,919.511782,1.0,11.000000,28.0,46.0,10681.0
interactive_controls_count,7368.0,12.703719,78.871701,0.0,0.000000,6.0,10.0,1282.0
named_interactive_controls,7368.0,12.423181,78.838897,0.0,0.000000,5.0,9.0,1282.0
unlabeled_controls_count,7368.0,0.280537,0.680462,0.0,0.000000,0.0,0.0,12.0
accessible_name_ratio,5141.0,0.977425,0.052407,0.0,0.964286,1.0,1.0,1.0
max_a11y_tree_depth,7368.0,6.234392,8.790815,0.0,4.000000,4.0,4.0,62.0


In [205]:
a11y_tree_features.to_csv(
    INTERIM_DIR
    / "a11y_tree_features.csv",
    index=False
)

In [206]:
m_assign.to_csv(
    INTERIM_DIR
    / "delfi_assign.csv",
    index=False
)

m_quiz.to_csv(
    INTERIM_DIR
    / "delfi_quiz.csv",
    index=False
)

In [207]:
irr_clean = irr.copy()

In [208]:
columnas_vacias = [
    col
    for col in irr_clean.columns
    if col.startswith("Unnamed")
    and irr_clean[col].isna().all()
]

columnas_vacias

['Unnamed: 11']

In [209]:
irr_clean = irr_clean.drop(
    columns=columnas_vacias
)

In [210]:
print(
    irr_clean.shape
)

irr_clean.head()

(104, 35)


,Name,Prototype Link,H1 Compare,H2 Compare,H3 Compare,H4 Compare,H5 Compare,H6 Compare,H7 Compare,H8 Compare,H9 Compare,H10 Compare,Unnamed: 12,Unnamed: 13,H1. Overal Rating (4),H1,H2. Overal Rating(4),H2,H3. Overal Rating(4),H3,H4 Overall Rating,H4,H5 Overall Rating,H5,H6 Overall Rating,H6,H7 Overall Rating,H7,H8 Overall Rating,H8,H9 Overall Rating,H9,H10 Overall Rating,H10,Unnamed: 0
0,Food Order,https://www.figma.com/design/OIOZZNZhiznIEalb5dpkMu/Food-Order-App-(Community)?node-id=0-1&p=f&t=Dx8Q70dOPPnSAAPO-0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gaira (Community app),https://www.figma.com/design/1WJTuXkM0OfRiWjZkD61n2/Gaia-Desktop-Design-(Community)?node-id=0-1&p=f&t=kawo30RBorPkO1d2-0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E-Tutor,https://www.figma.com/community/file/1271304360859402063,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tracking App,https://www.behance.net/gallery/198020739/Time-Tracker-mobile-application?tracking_source=project_owner_other_projects,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Travel,https://www.figma.com/community/file/1379689864793179441,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [211]:
irr_clean.to_csv(
    INTERIM_DIR
    / "irr_results.csv",
    index=False
)